In [1]:
# Import packages and initialize Earth Engine

import ee
import geemap
import pandas as pd
import numpy as np
import os
import seaborn as sns

geemap.ee_initialize()

### Initialize variables: Mask, extraction points and time frame 

In [13]:
date_start = ee.Date('2010-01-01') # year 2000
date_end = ee.Date('2010-12-31') # Orbital drift TERRA 2020-02-27, Orbital drift AQUA 2021-03-18

greenlandmask = ee.Image('OSU/GIMP/2000_ICE_OCEAN_MASK').select('ocean_mask').eq(0)
greenland = ee.Geometry.Polygon(
[[[-36.29516924635421, 83.70737243835941],
[-51.85180987135421, 82.75597137647488],
[-61.43188799635421, 81.99879137488564],
[-74.08813799635422, 78.10103528196419],
[-70.13305987135422, 75.65372336709613],
[-61.08032549635421, 75.71891096312955],
[-52.20337237135421, 60.9795530382023],
[-43.41430987135421, 58.59235996703347],
[-38.49243487135421, 64.70478286561182],
[-19.771731746354217, 69.72271161037442],
[-15.728762996354217, 76.0828635948066],
[-15.904544246354217, 79.45091003031243],
[-10.015872371354217, 81.62328742628017],
[-26.627200496354217, 83.43179828852398],
[-31.636966121354217, 83.7553561747887]]])

poi = ee.FeatureCollection("projects/ee-ivanburgov666/assets/randomGR5km_masked_260423")


In [19]:
# Load MODIS Terra and Aqua data, apply quality control and conversion functions
def lst_conversion(image):
    'Terra Night band selection and conversion'
    lst_night = image.select('LST_Night_1km').multiply(0.02).subtract(273.15).rename('LST_Night_C')
    lst_day = image.select('LST_Day_1km').multiply(0.02).subtract(273.15).rename('LST_Day_C')
    return image.addBands(lst_night).addBands(lst_day)

Terra = (
    ee.ImageCollection('MODIS/061/MOD11A1')
    .select(['LST_Night_1km', 'QC_Night', 'LST_Day_1km', 'QC_Day'])
    .filterDate(date_start, date_end)
    .filterBounds(greenland)
    .map(lst_conversion)
    .select(['LST_Night_C', 'QC_Night', 'LST_Day_C', 'QC_Day'])
)

Aqua = (
    ee.ImageCollection('MODIS/061/MYD11A1')
    .select(['LST_Night_1km', 'QC_Night','LST_Day_1km', 'QC_Day'])
    .filterDate(date_start, date_end)
    .filterBounds(greenland)
    .map(lst_conversion)
    .select(['LST_Night_C', 'QC_Night', 'LST_Day_C', 'QC_Day'])
)

imgTerra = Terra.toBands()
imgAqua = Aqua.toBands()

# Print number of bands in imgTerra
print('Number of bands in imgTerra:', imgTerra.bandNames().size().getInfo())

Number of bands in imgTerra: 1456


In [ ]:
def add_img_props(img):
    date = img.date().format('YYYY-MM-dd')
    return img.set('date', date)

def add_poi_props(f):
    return ee.Feature(f).set(img_props).set('sampleID', f.get('name'))

def extract(img):
    return img.reduceRegion(
        collection=poi,
        reducer= ee.Reducer.mean(),
        scale= 1000,
        tileScale= 16,
        crs= "EPSG:3411",
        geometry= greenland.geometry()
    ).map(add_img_props)


poiLST_terra = extract(imgTerra)
poiLST_aqua = extract(imgAqua)

geemap.ee_export_vector_to_drive(
collection=poiLST_terra,
folder="GEMLST_MODIS",
description ='poiTerra',
fileFormat='CSV'
)

Exporting poiTerra... Please check the Task Manager from the JavaScript Code Editor.


### Cloud Masking

### Export

In [ ]:
# # Create a result table
# def fc_to_df(fc):
#     'Convert a FeatureCollection to a Pandas DataFrame.'
#     features = fc.getInfo()['features']
#     dict_list = [f['properties'] for f in features]
#     df = pd.DataFrame(dict_list)
#     return df

# table_mod_day = fc_to_df(results_mod_day)


EEException: Collection query aborted after accumulating over 5000 elements.

In [8]:
# Export to Drive (CSV)
task = ee.batch.Export.table.toDrive(
    collection=results_mod_day,
    description='GEMLST_MOD_2003-2024',
    fileFormat='CSV',
    folder='gee'
)
task.start()

task2 = ee.batch.Export.table.toDrive(
    collection=results_mod_night,
    description='GEMLST_MOD_NIGHT_2003-2024',
    fileFormat='CSV',
    folder='gee'
)
task2.start()

task3 = ee.batch.Export.table.toDrive(
    collection=results_myd_day,
    description='GEMLST_MYD_DAY_2003-2024',
    fileFormat='CSV',
    folder='gee'
)
task3.start()

task4 = ee.batch.Export.table.toDrive(
    collection=results_myd_night,
    description='GEMLST_MYD_NIGHT_2003-2024',
    fileFormat='CSV',
    folder='gee'
)
task4.start()


ee.batch.Task.list()


[<Task SBO53TO3UNTTLHZSUYNK6OEP EXPORT_FEATURES: GEMLST_MYD_NIGHT_2003-2024 (READY)>,
 <Task YDDONWSRXWGBWSM3PIWU7SG6 EXPORT_FEATURES: GEMLST_MYD_DAY_2003-2024 (READY)>,
 <Task FIZT52VLV4RZVJUWCSCZPS5N EXPORT_FEATURES: GEMLST_MOD_NIGHT_2003-2024 (READY)>,
 <Task PYXFHC46PYIKIW72BWPN2OW6 EXPORT_FEATURES: GEMLST_MOD_2003-2024 (READY)>,
 <Task LBTOCCWKJWINWVPRC3EKG4H3 EXPORT_FEATURES: randomGR5km (READY)>,
 <Task 463UKQSCV3332WIWDCFMO5YP EXPORT_IMAGE: GEMLST_MODIS_20131231 (READY)>,
 <Task XJNDXQQXL47GTZOMLTF35YGG EXPORT_IMAGE: GEMLST_MODIS_20131230 (READY)>,
 <Task BGVXGNDPQSJR6GUST5RWNVXL EXPORT_IMAGE: GEMLST_MODIS_20131229 (READY)>,
 <Task QFPDIW6G7VBDFEDKSCYF5DB3 EXPORT_IMAGE: GEMLST_MODIS_20131228 (READY)>,
 <Task WL5I6EBQ2K3HQMN42VFYSGCL EXPORT_IMAGE: GEMLST_MODIS_20131227 (READY)>,
 <Task Q6UBY6KX7FDE7ZKY3OHUSY7K EXPORT_IMAGE: GEMLST_MODIS_20131226 (READY)>,
 <Task APEFYBTF3ZSGF7COGAGWJMZG EXPORT_IMAGE: GEMLST_MODIS_20131225 (READY)>,
 <Task VTTS5GAA7AD5DKYFEDELYUUJ EXPORT_IMAGE: G